### cassio is the official Python client library used to connect LangChain → AstraDB / Cassandra database.

In [22]:
# NEW LangChain imports (2025)
from langchain_huggingface import HuggingFaceEmbeddings     # FREE embeddings
from langchain_community.vectorstores import FAISS          # Local vector DB
from langchain_text_splitters import RecursiveCharacterTextSplitter  # Updated splitter
from langchain_groq import ChatGroq                         # Groq LLM
from langchain_community.vectorstores.cassandra import Cassandra  # Cassandra DB
import cassio                                               # Cassandra/Astra client
from datasets import load_dataset


In [23]:
from PyPDF2 import PdfReader

### SETUP

In [24]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [25]:
ASTRA_DB_APPLICATION_TOKEN = os.getenv("ASTRA_DB_APPLICATION_TOKEN")
ASTRA_DB_ID = os.getenv("ASTRA_DB_ID")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")


In [26]:
pd = PdfReader("Union_Budget_2023.pdf")

In [27]:
from typing_extensions import Concatenate
raw_text = " "
for i , page in enumerate(pd.pages):
    content = page.extract_text()
    if content:
        raw_text += content

In [28]:
raw_text

"  \n \nFebruary 1, 202 3 \nPRS Legislative Research  ◼ Institute for Policy Research Studies  \n3rd Floor, Gandharva Mahavidyalaya ◼ 212, Deen Dayal Upadhyaya Marg ◼ New Delhi – 110002 \nTel: (011) 43434035, 23234801 ◼ www.prsindia.org  \n \nUnion  Budget 2023-24 Analysis  \nBudget Highlights   \n▪ Expenditure: The government proposes to spend Rs 45,03,097 crore in 202 3-24, which is an increase  of 7.5% over the \nrevised estimate of 202 2-23.  In 202 2-23, total expenditure is estimated to be 6.1%  higher than the budget estimate.  \n▪ Receipts: The receipts (other than borrowings) in 202 3-24 are expected to be to Rs 27,16,281 crore,  an increase of \n11.7% over revised estimate of 202 2-23.  In 202 2-23, total receipts (other than borrowings) are estimated to be 6.5% \nhigher than the budget estimates.  \n▪ GDP : The government has estimated a nominal GDP growth rate of 10.5% in 202 3-24 (i.e., real growth plus inflation).  \n▪ Deficits: Revenue deficit in 2023 -24 is targeted at 

### Initialize Connection with DB

In [29]:
cassio.init(token=ASTRA_DB_APPLICATION_TOKEN, database_id=ASTRA_DB_ID)

### Create LangChain Embeddings and LLM objects or later usage

In [30]:
# LLM using Groq
llm = ChatGroq(api_key=GROQ_API_KEY, model="llama-3.1-8b-instant")
# FREE local embeddings
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

### Create My LangChain Vectore Store ...... backed by Astra DB

In [31]:
astra_vector_store = Cassandra(embedding=embedding, table_name="QA_mini", session=None, keyspace=None)

In [32]:
from langchain_text_splitters import CharacterTextSplitter
text_splitter = CharacterTextSplitter(separator="\n", chunk_size=800, chunk_overlap=200, length_function=len)
texts = text_splitter.split_text(raw_text)

In [33]:
texts[:50]

['February 1, 202 3 \nPRS Legislative Research  ◼ Institute for Policy Research Studies  \n3rd Floor, Gandharva Mahavidyalaya ◼ 212, Deen Dayal Upadhyaya Marg ◼ New Delhi – 110002 \nTel: (011) 43434035, 23234801 ◼ www.prsindia.org  \n \nUnion  Budget 2023-24 Analysis  \nBudget Highlights   \n▪ Expenditure: The government proposes to spend Rs 45,03,097 crore in 202 3-24, which is an increase  of 7.5% over the \nrevised estimate of 202 2-23.  In 202 2-23, total expenditure is estimated to be 6.1%  higher than the budget estimate.  \n▪ Receipts: The receipts (other than borrowings) in 202 3-24 are expected to be to Rs 27,16,281 crore,  an increase of \n11.7% over revised estimate of 202 2-23.  In 202 2-23, total receipts (other than borrowings) are estimated to be 6.5%',
 '11.7% over revised estimate of 202 2-23.  In 202 2-23, total receipts (other than borrowings) are estimated to be 6.5% \nhigher than the budget estimates.  \n▪ GDP : The government has estimated a nominal GDP growth rat

### Load the dataset into the vector store

### What Is a Retriever?

A **retriever** is a simple interface that:

- Fetches **the most relevant text chunks** from your vector database  
- Returns them to the LLM (Groq, OpenAI, etc.)  
- Helps the model answer questions based on **your documents**, not its training data  


### How a Retriever Works Internally

1. User asks:  
   “Summarize the PDF.”

2. Retriever embeds the query → vector  
3. Compares it with stored document vectors (FAISS, AstraDB, Cassandra, Chroma, Pinecone, etc.)  
4. Returns the most relevant chunks  
5. LLM reads the chunks  
6. LLM generates the final answer  

In [34]:
astra_vector_store.add_texts(texts[:50])

print("Inserted %i headlines." % len(texts[:50]))

# astra_vector_index = VectorStoreIndexWrapper(vectorstore = astra_vector_store) -> deprecated 
retriever = astra_vector_store.as_retriever()


Inserted 50 headlines.


In [35]:
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are an expert system that answers user questions strictly based on retrieved document chunks.

***Instructions:***
- Use only the information provided in the context.
- Do NOT guess or hallucinate.
- If the answer is not present, respond with:
  "The answer is not available in the provided document."
- Provide a precise, well-structured explanation.

***Context Chunks:***
{context}

***User Query:***
{input}

***Final Answer:***
""")


# 1. Build document-combining chain
combine_docs_chain = create_stuff_documents_chain(llm = llm, prompt=prompt)

# 2. Build retrieval chain
qa = create_retrieval_chain(
    retriever,
    combine_docs_chain
)

# 3. Run query
# result = qa.invoke({"query": "What is this PDF about?"})
result = qa.invoke({"input": "What is this PDF about?"})
print(result["answer"])


The document is about the analysis of the Union Budget 2023-24. It contains information about the budget estimates for 2023-24 as compared to the revised estimates of 2022-23, along with certain policy announcements and changes in governance. 

Specifically, it mentions the increase in the senior citizens savings scheme from Rs 15 lakh to Rs 30 lakh, and several governance-related initiatives such as simplification of the KYC process, implementation of a Unified Filing Process, and the launch of a Voluntary Settlement Scheme.


In [36]:
# first_question = True
# while True:
#     if first_question:
#         query_text = input("\nEnter your question (or type 'quit' to exit): ").strip()
#     else:
#         query_text = input("\nWhat's your next question (or type 'quit' to exit): ").strip()

#     if query_text.lower() == "quit":
#         break

#     if query_text == "":
#         continue

#     first_question = False

#     print("\nQUESTION: \"%s\"" % query_text)
#     # answer = astra_vector_index.query(query_text, llm=llm).strip() -> not supported -> we will use 
#     result = qa.invoke({"input": query_text})
#     answer = result["answer"]          # answer string from the chain
    
#     print("ANSWER: \"%s\"\n" % answer)

#     print("FIRST DOCUMENTS BY RELEVANCE:")
#     for doc, score in astra_vector_store.similarity_search_with_score(query_text, k=4):
#         print("        [%0.4f] \"%s ...\"" % (score, doc.page_content[:84]))

first_question = True
while True:
    if first_question:
        query_text = input("\nEnter your question (or type 'quit' to exit): ").strip()
    else:
        query_text = input("\nWhat's your next question (or type 'quit' to exit): ").strip()

    if query_text.lower() == "quit":
        break

    if query_text == "":
        continue

    first_question = False

    print("\nQUESTION: \"%s\"" % query_text)

    # Get answer using your RAG chain
    result = qa.invoke({"input": query_text})
    answer = result["answer"]

    # Print ONLY the answer
    print("ANSWER: \"%s\"\n" % answer)



QUESTION: "1. What are the changes in the new income tax regime?"
ANSWER: "The answer is available in the provided document.

Changes in the new income tax regime include:

- The number of tax slabs has been reduced from six to five.
- The surcharge on the income when it exceeds Rs 5 crore will be reduced from 37% to 25%.
- Those with income up to Rs 5 lakh can avail a rebate and not pay any taxes; this limit has been raised to Rs 7 lakh.
- The standard deduction will be available under the new tax regime."


QUESTION: "2. What is the standard deduction available under the new tax regime?"
ANSWER: "The standard deduction will be available under the new tax regime."




QUESTION: "what are the questions that can be asked from this pdf ?"
ANSWER: "Based on the provided context chunks, the following questions can be asked:

1. What are the changes in the new income tax regime?
   - Answer: The number of tax slabs has been reduced from six to five, and the surcharge on income when it exceeds Rs 5 crore will be reduced from 37% to 25%.

2. How many tax slabs are there in the new income tax regime?
   - Answer: There are five tax slabs in the new income tax regime.

3. What is the limit for income up to which no tax is payable?
   - Answer: The limit has been raised to Rs 7 lakh.

4. What is the standard deduction available under the new tax regime?
   - Answer: The standard deduction will be available under the new tax regime.

5. What are the legislative proposals mentioned in the policy highlights?
   - Answer: Amendments will be made to the Banking Regulation Act, 1949, and Banking Companies.

6. What is the current limit for income up to which a rebate is available?
   - Answer: The current limit is Rs 5 lakh.

7. What is the proposed change in the surcharge on income exceeding Rs 5 crore?
   - Answer: The surcharge will be reduced from 37% to 25%.